In [ ]:
from chemrar_retro import retro, config, scores, analyze
from pydantic import BaseModel
import pandas as pd
import numpy as np
from pathlib import Path

e:\GitHub\ChemRAR\.venv\Lib\site-packages\BRSAScore\BRSAScore.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


# Проблемы с оберткой _aizynthfinder_


## Описание


Оригинальный `aizynthfinder.aizynthfinder.AiZynthFinder` - по сути своей, набор функций вокруг базового класса `aizynthfinder.context.config.Configuration`. Этот класс собирается при чтении файла или сборки из словаря и передаётся целиком в каждый компонент `AiZynthFinder`. К сожалению, внутри `Configuration` содержатся не только неизменяемые объекты, но и списки, используемых в данный момент, стоков, скореров и тд, а также их кэш.

Чтобы параллельные потоки не создавали гонку и не искажали вычисления друг друга, нужно превратить `AiZynthFinder` В чистую функцию, без сайд-эффектов.


## Решение


Чтобы проверить наличие сайд-эффектов был написан класс `Snapshot`, который рекурсивно делает снимок всех внутренних компонентов до определённого уровня. (Чтобы проверка была быстрой, он не проверяет значения, если в списке или в словаре оказывается слишком много данных)


In [ ]:
# snapshot
from typing import Any, Iterator
import pickle

from typing import Hashable


class Snapshot:
    def __init__(self, obj, *, depth: int, file: Path | None = None) -> None:
        self.depth = depth
        self.dump_file = file

        if file:
            with open(file.as_posix(), "wb") as f:
                pickle.dump(
                    Snapshot._to_image(obj, depth),
                    file=f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )
            self.dump = None
        else:
            self.dump = pickle.dumps(
                Snapshot._to_image(obj, depth),
                protocol=pickle.HIGHEST_PROTOCOL,
            )

    def get_image(self):
        return self._get_image()

    def diff(self, obj, max_depth: int | None = None) -> dict[str, tuple]:
        depth = min(self.depth, max_depth or self.depth)
        a = self._get_image()

        diff = Snapshot._snapshot_diff(a, Snapshot._to_image(obj, max_depth=self.depth))
        result = {}
        for i, x0, x1 in diff:
            title = []

            for t in i.replace("]", "").split("[")[: depth + 1]:
                if t == "":
                    continue
                if t.startswith("'."):
                    title.append(t.replace("'", ""))
                else:
                    title.append(f"[{t}]")

            key = "".join(title).strip()

            # key = "".join(
            #                 f"['{t}']" if t and not t.startswith(".") else t

            #             ).strip()

            result.update({key: (x0, x1)})

        return result

    def _get_image(self) -> dict:  # type: ignore
        if self.dump_file:
            with open(self.dump_file, "rb") as f:
                return pickle.load(f)
        elif self.dump:
            return pickle.loads(self.dump)

    @staticmethod
    def _to_image(
        obj: object,
        max_depth: int,
        depth: int = 0,
        memo: frozenset[int] = frozenset(),
    ) -> object:

        if depth > max_depth:
            return _DepthLimited(type(obj).__name__)

        if id(obj) in memo:
            return _Skipped("cycle")

        if isinstance(obj, dict):
            memo = memo | {id(obj)}
            if len(obj) > 500:
                return _Skipped("long dict")

            return {
                k if isinstance(k, Hashable) else "@nonHashable@": Snapshot._to_image(
                    v, max_depth, depth + 1, memo
                )
                if isinstance(k, str)
                else (
                    Snapshot._to_image(k, max_depth, depth + 1, memo),
                    Snapshot._to_image(v, max_depth, depth + 1, memo),
                )
                for k, v in obj.items()
            }

        if isinstance(obj, (list, tuple, set)):
            memo = memo | {id(obj)}
            if len(obj) > 500:

                return _Skipped("long sequence")
            return [Snapshot._to_image(v, max_depth, depth + 1, memo) for v in obj]

        if isinstance(obj, (pd.DataFrame, pd.Series, np.ndarray)):
            return _Skipped("data")


        attrs: dict[str, object] = {}
        if hasattr(obj, "__dict__"):
            attrs.update(vars(obj))
        for name in getattr(type(obj), "__slots__", ()):
            if hasattr(obj, name):
                attrs[name] = getattr(obj, name)

        if attrs:
            return {
                "__type__": type(obj).__name__,
                **{
                    f".{k}": Snapshot._to_image(v, max_depth, depth + 1, memo)
                    for k, v in attrs.items()
                },
            }

        return repr(obj)

    @staticmethod
    def _snapshot_diff(a: Any, b: Any, path: str = "") -> Iterator[tuple[str, Any, Any]]:
        if type(a) is not type(b):
            yield path, a, b
            return

        if isinstance(a, dict):
            for key in a.keys() | b.keys():
                if key not in a:
                    yield f"{path}[{key!r}]", "<missing>", b[key]
                elif key not in b:
                    yield f"{path}[{key!r}]", a[key], "<missing>"
                else:
                    yield from Snapshot._snapshot_diff(a[key], b[key], f"{path}[{key!r}]")
            return

        if isinstance(a, (list, tuple)):
            if len(a) != len(b):
                yield f"{path}.len", len(a), len(b)
                return
            for i, (av, bv) in enumerate(zip(a, b)):
                yield from Snapshot._snapshot_diff(av, bv, f"{path}[{i}]")
            return

        if isinstance(a, set):
            if a != b:
                yield path, a - b, b - a
            return
        try:
            if a != b:
                yield path, a, b
        except:
            if all(a != b):
                yield path, a, b


class _Skipped:
    __slots__ = ("type_name",)

    def __init__(self, type_name: str) -> None:
        self.type_name = type_name

    def __repr__(self) -> str:
        return f"<skipped:{self.type_name}>"

    def __hash__(self) -> int:
        return hash(self.type_name + "_Skipped")

    def __eq__(self, other: object) -> bool:
        return isinstance(other, _Skipped) and self.type_name == other.type_name


class _DepthLimited:
    __slots__ = ("type_name",)

    def __init__(self, type_name: str) -> None:
        self.type_name = type_name

    def __repr__(self) -> str:
        return f"<depth-limit:{self.type_name}>"

    def __hash__(self) -> int:
        return hash(self.type_name + "_DepthLimited")

    def __eq__(self, other: object) -> bool:
        return isinstance(other, _DepthLimited) and self.type_name == other.type_name


In [3]:
class A(BaseModel):
    a: int = 1
    b: dict = dict(x=0, y=8)
    c: list = [1, 2]


a1 = A()
a2 = A(
    a=1,
    b=dict(x=0, y=7),
    c=[1, 3],
)

s1 = Snapshot(a1, depth=10)
diff = s1.diff(a2)

print("\n".join(f"{e:3}|{i}" for e, i in enumerate(diff)))

  0|.__pydantic_fields_set__.len
  1|.b['y']
  2|.c[1]
  3|.__dict__['c'][1]
  4|.__dict__['b']['y']


In [ ]:
engine = retro.create_engine(
    expansion_policies=dict(
        uspto=config.ExpansionPolicy(
            model=Path("data/model/uspto_model.onnx"),
            template=Path("data/model/uspto_templates.csv.gz"),
        ),
        ringbreaker=config.ExpansionPolicy(
            model=Path("data/model/uspto_ringbreaker_model.onnx"),
            template=Path("data/model/uspto_ringbreaker_templates.csv.gz"),
        ),
    ),
    filters=dict(
        uspto=config.FilterPolicy(model=Path("data/model/uspto_filter_model.onnx")),
    ),
    stock=dict(
        zinc=config.Stock(path=Path("data/model/zinc_stock.hdf5")),
    ),
)

smiles = "Cc1cccc(C)c1N(CC(=O)Nc1ccc(-c2ncon2)cc1)C(=O)C1CCS(=O)(=O)CC1"

Предельный уровень копирования =8. Дальше внутри библиотеки находится `Lock`,


In [5]:
snap = Snapshot(engine,depth=8)

## Тест функции копирования


Решение проблемы гонки - скопировать все лёгкие компоненты, которые меняются при применении. Это делает функция `retro._copy_engine`, она просто копирует все внутренние части стоков, скореры, стратегии фильтрации и расширения вместе с их кешами, но при этом не трогает веса моделей.


In [6]:
def imutate_func(
    engine: retro.Engine,
    smiles: str,
):
    engine = retro._copy_engine(engine)

    engine.target_smiles = smiles
    engine.expansion_policy.select_all()
    engine.tree_search()
    engine.build_routes()
    return

In [7]:
imutate_func(engine,smiles=smiles )

В основные компоненты не входит логер. Конечно, его стоит скопировать, но для задачи тестового задания этого достаточно.


In [8]:
diff = snap.diff(engine)
diff_keys = [k for k in diff if "logger" not in k]

print("\n".join(f"{e:3}|{i}" for e, i in enumerate(diff_keys)))

In [9]:
diff.keys()

dict_keys(['._logger._cache[10]', '._logger._cache[20]', "._logger.manager.loggerDict['aizynthfinder']._cache[10]", "._logger.manager.loggerDict['aizynthfinder']._cache[20]", "._logger.manager.loggerDict['aizynthfinder'].level", "._logger.manager.loggerDict['rdkit.Chem'].loggerMap[<Logger rdkit.Chem.MolStandardize.normalize (WARNING)>][0]._cache", "._logger.manager.loggerDict['rdkit.Chem'].loggerMap[<Logger rdkit.Chem.MolStandardize.charge (WARNING)>][0]._cache", "._logger.manager.loggerDict['rdkit.Chem.MolStandardize.normalize']._cache[10]", "._logger.manager.loggerDict['rdkit'].handlers[0].stream.write", "._logger.manager.loggerDict['rdkit.Chem.MolStandardize.charge']._cache[10]", '._logger.level', '.filter_policy._logger._cache[10]', '.filter_policy._logger._cache[20]', ".filter_policy._logger.manager.loggerDict['aizynthfinder']._cache[10]", ".filter_policy._logger.manager.loggerDict['aizynthfinder']._cache[20]", ".filter_policy._logger.manager.loggerDict['aizynthfinder'].level", ".

## Тест работы


# Диапазон и монотонность скоринг функций


## Описание


Проблема заключается в том, что функции в библиотеке aizynthfinder неконсистентны - их нельзя просто скомбинировать вместе.

1. Средняя арифметическое, softmax и тд нельзя взять, так как значения каждой функции находятся в разных диапазонах и, следовательно, несопоставимы.
2. Нужно учитывать прямой и обратный порядок в зависимости от того, где используется функция:
   1. обычный скорринг используется параметр - **\_reverse_order**, который меняет направление сортировки при применении скоррера на набор узлов или всё дерево.
   2. комбинированный скорринг - нужно обязательно вводить инвертированный масштабатор, так как параметр **\_reverse_order** используется только при сортировке, но не при вычислении (что не логично)
   3. Нет валидации выхода, поэтому можно получить случайно отрицательное число, есть ли писать свою функцию. Существует масштабатор, который просто возводит в степень 0.98 (по умолчанию), что для отрицательного числа выдаёт комплексное.
3. Нельзя удобно задать параметры функции и масштабатора до создания движка.

**Возрастающая функция (0-плохо, 1-отлично)**

```python
class FractionInStockScorer(Scorer):
    """Class for scoring nodes based on the fraction in stock"""

    scorer_name = "fraction in stock"

    def __init__(
        self, config: Configuration, scaler_params: Optional[StrDict] = None
    ) -> None:
        super().__init__(config, scaler_params)
        # This is necessary because config should not be optional for this scorer
        self._config: Configuration = config

    def _score_node(self, node: MctsNode) -> float:
        num_in_stock = np.sum(node.state.in_stock_list)
        num_molecules = len(node.state.mols)
        return float(num_in_stock) / float(num_molecules)

    def _score_reaction_tree(self, tree: ReactionTree) -> float:
        leaves = list(tree.leafs())
        num_in_stock = sum(mol in self._config.stock for mol in leaves)
        num_molecules = len(leaves)
        return float(num_in_stock) / float(num_molecules)
```

**Убывающая функция (0-идеально, inf - плохо)**

```python
class NumberOfPrecursorsScorer(Scorer):
    """Class for scoring nodes based on the number of pre-cursors in a node or route"""

    scorer_name = "number of pre-cursors"

    def __init__(
        self,
        config: Optional[Configuration] = None,
        scaler_params: Optional[StrDict] = None,
    ) -> None:
        super().__init__(config, scaler_params)
        self._reverse_order = False

    def _score_node(self, node: MctsNode) -> float:
        return len(node.state.mols)

    def _score_reaction_tree(self, tree: ReactionTree) -> float:
        return len(list(tree.leafs()))
```


## Решение


Чтобы не сломать существующее поведение, можно просто проверять значения при создании кастомных скорреров. И предупреждать, если значения могут изменять направления (например, указать k>0 для NumberOfPrecursorsScorer, который всегда требует k<0 если мы его комбинируем с чем-то другим)

Для этого используется собственный класс-обёртка `Score`. Инициализация которого не требует объявления движка и содержит в себе класс нужного скоррера.


In [2]:
engine = retro.create_engine(
    expansion_policies=dict(
        uspto=config.ExpansionPolicy(
            model=Path("download/example/uspto_model.onnx"),
            template=Path("download/example/uspto_templates.csv.gz"),
        ),
        ringbreaker=config.ExpansionPolicy(
            model=Path("download/example/uspto_ringbreaker_model.onnx"),
            template=Path("download/example/uspto_ringbreaker_templates.csv.gz"),
        ),
    ),
    filters=dict(
        uspto=config.FilterPolicy(model=Path("download/example/uspto_filter_model.onnx")),
    ),
    stock=dict(
        zinc=config.Stock(path=Path("download/example/zinc_stock.hdf5")),
    ),
    scores=[scores.availability.NPrecursors()],
)

C:\Users\krayn\AppData\Local\Temp\ipykernel_2380\1182536617.py:18: UserWarning: Score reversed - use scaler with up=False
  scores=[scores.availability.NPrecursors()],


In [10]:
engine.scorers.items

['state score',
 'number of reactions',
 'number of pre-cursors',
 'number of pre-cursors in stock']

In [ ]:
engine_selected = retro.select(
    engine,
    stocks=engine.stock.items,
    filter_policies=engine.filter_policy.items,
    search_scorers={"state score": 1},
    expansion_policies=engine.expansion_policy.items,
)
tree = retro.generate_tree(engine_selected, smiles)
retro.search_tree(tree, 100)
result = ana.analyze_tree(tree, scorers=["number of reactions", "state score"], top_n=10)